 #### Week 13: Creating ADaM and SDTM Vital Signs (VS) Synthetic Datasets and Merging Them

Here's an example of SAS code for creating synthetic datasets for ADaM (Analysis Data Model) and SDTM (Study Data Tabulation Model) for ADaM and SDTM datasets. This is a simple example to get you started, and you can modify it based on the specific structure and requirements of your clinical data.

In [ ]:
/* Create Subject Demographics (DM) */
data sdm_dm;
   input USUBJID $ 1-3 AGE SEX $;
   datalines;
   001 34 M
   002 45 F
   003 29 M
   004 38 F
   ;
run;

In [ ]:

/* Create Adverse Events (AE) */
data sdm_ae;
   infile datalines dlm=',';
   input USUBJID :$3.  AEDECOD :$9. AESTDTC :yymmdd10. AESEV :$8.;
   datalines;
   001,Headache,2023-01-01,Mild
   002,Nausea,2023-01-02,Moderate
   003,Fatigue,2023-01-03,Severe
   004,Dizziness,2023-01-04,Mild
   ;
run;

In [ ]:
/* Create Laboratory Data (LB) */

data sdm_lb;
   input USUBJID $  LBTPT $ LBSTRESN LBSTRESU $ EVALUATION $;
   datalines;
   001 Pre 10 mg/dL Normal
   002 Pre 12 mg/dL Normal
   003 Pre 15 mg/dL High
   004 Post 20 mg/dL High
   ;
run;


In [ ]:
/* Create Randomized Treatment Data (TRT) */
data sdm_trt;
   input USUBJID $ TRTSDT :yymmdd10. TRTEDT :yymmdd10.;
   format TRTSDT TRTEDT yymmdd10.;
   datalines;
   001 2023-01-01 2023-03-01
   002 2023-01-05 2023-03-10
   003 2023-01-10 2023-03-15
   004 2023-01-15 2023-03-20
   ;
run;


In [ ]:
/* Create Analysis Dataset for Adverse Events (ADAE) */
data adam_adae;
   set sdm_ae;
   if AESEV = 'Severe' then AESEV = '1';
   else if AESEV = 'Moderate' then AESEV = '2';
   else AESEV = '3';
run;


In [ ]:

/* Create Analysis Dataset for Laboratory Results (ADLB) */
data adam_adlb;
   set sdm_lb;
   if LBSTRESN > 15 then LBSTRESU = 'High';
   else LBSTRESU = 'Normal';
run;



In [ ]:
/* Create Subject-level Analysis Data (ADSL) */
data adam_adsl;
   set sdm_dm;
   if SEX = 'M' then SEXFL = '1';
   else SEXFL = '0';
run;

Explanation:

SDTM Datasets (sdm_*): These are synthetic datasets based on SDTM standards, including subject demographics (DM), adverse events (AE), laboratory results (LB), and randomized treatment data (TRT).

ADaM Datasets (adam_*): These datasets are derived from SDTM datasets and are structured for analysis. For example:

ADAE (Adverse Event Analysis): Converts the severity of adverse events into numeric codes.
ADLB (Laboratory Analysis): Flags laboratory results as "High" or "Normal" based on predefined thresholds.
ADSL (Subject-Level Analysis): Creates a subject-level analysis dataset with additional variables, such as a flag for male subjects.
You can extend these datasets with more variables and complexity based on your actual study design and data requirements.

Explanation of the Extended Datasets:
SDTM Datasets (sdm_*):

sdm_dm: Added more subjects with additional demographic information like race and ethnicity.
sdm_ae: Added a severity (AESEV) and relationship to treatment (AEREL).
sdm_lb: Added more laboratory data with tests like Hemoglobin and Cholesterol and a new variable LBTEST to track the test type.
sdm_trt: Added a new treatment group (TRTGROUP).
ADaM Datasets (adam_*):

adam_adae: Added an analysis flag for whether the adverse event is related to the treatment (AERELFL).
adam_adlb: Added a lab test result flag (LBTESTFL) to indicate if the lab test is related to Hemoglobin.
adam_adsl: Introduced an age group classification (AGEGRP).
adam_adtrt: Created treatment flags to indicate the treatment group for analysis.
Final ADaM Dataset (adam_final):

Combined all individual ADaM datasets (subject, adverse events, lab results, and treatment) into one final dataset adam_final.

In [8]:
/* Create Subject Demographics (DM) */
ods html close;
options nodate nosource notes nonumber;
data sdm_dm;
   input USUBJID $ AGE SEX $ RACE $ ETHNIC $;
   datalines;
   001 34 M Asian Hispanic
   002 45 F White Non-Hispanic
   003 29 M Black Non-Hispanic
   004 38 F Hispanic Non-Hispanic
   005 40 M White Hispanic
   006 33 F Black Hispanic
   007 50 M Asian Non-Hispanic
   008 60 F White Non-Hispanic
   ;
run;

                                                           The SAS System


NOTE: The data set WORK.SDM_DM has 8 observations and 5 variables.
NOTE: DATA statement used (Total process time):
      real time           0.02 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000010


In [10]:
ods html close;
options nodate nosource notes nonumber;
data sdm_ae;
   length USUBJID $3 AEDECOD $9 c_AESTDTC $10 AESEV $9 AEREL $12;
   input USUBJID $ AEDECOD $ c_AESTDTC $ AESEV $ AEREL $;
   AESTDTC = input(c_AESTDTC, yymmdd10.); /* Convert to SAS date */
   format AESTDTC yymmdd10.; /* Ensure date format */
   drop c_AESTDTC; /* Remove the character version */
   datalines;
001 Headache 2023-01-01 Mild Related
002 Nausea 2023-01-02 Moderate Not Related
003 Fatigue 2023-01-03 Severe Related
004 Dizziness 2023-01-04 Mild Not Related
005 Vomiting 2023-01-05 Severe Related
006 Rash 2023-01-06 Mild Not Related
007 Headache 2023-01-07 Moderate Related
008 Nausea 2023-01-08 Severe Related
;
run;


                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: The data set WORK.SDM_AE has 8 observations and 5 variables.
NOTE: DATA statement used (Total process time):
      real time           0.01 seconds
      cpu time            0.00 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000012


In [12]:
/* Create Laboratory Data (LB) */
ods html close;
options nodate nosource notes nonumber;
data sdm_lb;
   length LBTEST $11;
   input USUBJID $ LBTPT $ LBSTRESN LBSTRESU $ LBTEST $ ;
   datalines;
   001 Pre 10 mg/dL Normal Hemoglobin
   002 Pre 12 mg/dL Normal Hemoglobin
   003 Pre 15 mg/dL High Hemoglobin
   004 Post 20 mg/dL High Hemoglobin
   005 Pre 80 mg/dL Normal Cholesterol
   006 Pre 90 mg/dL High Cholesterol
   007 Post 70 mg/dL Normal Cholesterol
   008 Post 60 mg/dL Normal Cholesterol
   ;
run;

                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: The data set WORK.SDM_LB has 8 observations and 5 variables.
NOTE: DATA statement used (Total process time):
      real time           0.00 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000014


In [14]:
/* Create Randomized Treatment Data (TRT) */
ods html close;
options nodate nosource notes nonumber;
data sdm_trt;
   infile datalines dlm = ' ';
   input USUBJID $ TRTSDT : yymmdd10. TRTEDT : yymmdd10. TRTGROUP $;
   format TRTSDT TRTEDT yymmdd10.;
   datalines;
   001 2023-01-01 2023-03-01 DrugA
   002 2023-01-05 2023-03-10 DrugB
   003 2023-01-10 2023-03-15 DrugA
   004 2023-01-15 2023-03-20 DrugB
   005 2023-01-20 2023-03-25 DrugA
   006 2023-01-25 2023-03-30 DrugB
   007 2023-01-30 2023-04-01 DrugA
   008 2023-02-05 2023-04-10 DrugB
   ;
run;

                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: The data set WORK.SDM_TRT has 8 observations and 4 variables.
NOTE: DATA statement used (Total process time):
      real time           0.08 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000016


In [16]:
/* Create Analysis Dataset for Adverse Events (ADAE) */
ods html close;
options nodate nosource notes nonumber;
data adam_adae;
   set sdm_ae;
   if AESEV = 'Severe' then AESEV = '1';
   else if AESEV = 'Moderate' then AESEV = '2';
   else AESEV = '3';
   /* Create analysis flags for related and severity */
   if AEREL = 'Related' then AERELFL = 'Y';
   else AERELFL = 'N';
run;

/* Create Analysis Dataset for Laboratory Results (ADLB) */
data adam_adlb;
   set sdm_lb;
   if LBSTRESN > 15 then LBSTRESU = 'High';
   else LBSTRESU = 'Normal';
   /* Add lab test result flag */
   if LBTEST = 'Hemoglobin' then LBTESTFL = '1';
   else LBTESTFL = '0';
run;

/* Create Subject-level Analysis Data (ADSL) */
data adam_adsl;
   set sdm_dm;
   /* Gender flag */
   if SEX = 'M' then SEXFL = '1';
   else SEXFL = '0';
   /* Age group flag */
   if AGE < 40 then AGEGRP = 'Young';
   else if AGE >= 40 and AGE < 60 then AGEGRP = 'Middle-aged';
   else AGEGRP = 'Senior';
run;

/* Extended Analysis Dataset for Treatment (ADTRT) */
data adam_adtrt;
   set sdm_trt;
   /* Treatment group flag */
   if TRTGROUP = 'DrugA' then TRTFL = '1';
   else if TRTGROUP = 'DrugB' then TRTFL = '2';
run;

/* Combining Multiple Datasets for a Final ADaM Analysis Dataset */
data adam_final;
   merge adam_adsl(in=a) adam_adae(in=b) adam_adlb(in=c) adam_adtrt(in=d);
   by USUBJID;
   if a and b and c and d;
run;



                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: There were 8 observations read from the data set WORK.SDM_AE.
NOTE: The data set WORK.ADAM_ADAE has 8 observations and 6 variables.
NOTE: DATA statement used (Total process time):
      real time           0.02 seconds
      cpu time            0.01 seconds
      


NOTE: There were 8 observations read from the data set WORK.SDM_LB.
NOTE: The data set WORK.ADAM_ADLB has 8 observations and 6 variables.
NOTE: DATA statement used (Total process time):
      real time           0.09 seconds
      cpu time            0.04 seconds
      


NOTE: There were 8 observations read from the data set WORK.SDM_DM.
NOTE: The data set WORK.ADAM_ADSL has 8 observations and 7 variables.
NOTE: DATA statement used (Total process time):
      real time           0.06 seconds
      cpu time            0.00 seconds
      


NOTE: There were 8 observations read from the da

Explanation of the Extended Datasets:
SDTM Datasets (sdm_*):

sdm_dm: Added more subjects with additional demographic information like race and ethnicity.
sdm_ae: Added a severity (AESEV) and relationship to treatment (AEREL).
sdm_lb: Added more laboratory data with tests like Hemoglobin and Cholesterol and a new variable LBTEST to track the test type.
sdm_trt: Added a new treatment group (TRTGROUP).
ADaM Datasets (adam_*):

adam_adae: Added an analysis flag for whether the adverse event is related to the treatment (AERELFL).
adam_adlb: Added a lab test result flag (LBTESTFL) to indicate if the lab test is related to Hemoglobin.
adam_adsl: Introduced an age group classification (AGEGRP).
adam_adtrt: Created treatment flags to indicate the treatment group for analysis.
Final ADaM Dataset (adam_final):

Combined all individual ADaM datasets (subject, adverse events, lab results, and treatment) into one final dataset adam_final.
The data sets have been enriched with additional variables, new records, and flags to simulate a more complex clinical trial structure.









In [18]:
/* Creating Extended Synthetic SDTM Data - Subject Demographics (DM) */
ods html close;
options nodate nosource notes nonumber;
data sdm_dm;
   input USUBJID $ AGE SEX $ RACE $ ETHNIC $;
   datalines;
   001 34 M Asian Hispanic
   002 45 F White Non-Hispanic
   003 29 M Black Non-Hispanic
   004 38 F Hispanic Non-Hispanic
   005 40 M White Hispanic
   006 33 F Black Hispanic
   007 50 M Asian Non-Hispanic
   008 60 F White Non-Hispanic
   ;
run;



                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: The data set WORK.SDM_DM has 8 observations and 5 variables.
NOTE: DATA statement used (Total process time):
      real time           0.03 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000020


In [20]:
/* Create Adverse Events (AE) */
ods html close;
options nodate nosource notes nonumber;
data sdm_ae;
   input USUBJID $ AEDECOD $ AESTDTC: yymmdd10.  AESEV $ AEREL $;
   datalines;
   001 Headache 2023-01-01 Mild Related
   002 Nausea 2023-01-02 Moderate Not Related
   003 Fatigue 2023-01-03 Severe Related
   004 Dizziness 2023-01-04 Mild Not Related
   005 Vomiting 2023-01-05 Severe Related
   006 Rash 2023-01-06 Mild Not Related
   007 Headache 2023-01-07 Moderate Related
   008 Nausea 2023-01-08 Severe Related
   ;
run;


                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: The data set WORK.SDM_AE has 8 observations and 5 variables.
NOTE: DATA statement used (Total process time):
      real time           0.01 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000022


In [22]:
/* Create Laboratory Data (LB) */
ods html close;
options nodate nosource notes nonumber;
data sdm_lb;
   input USUBJID $ LBTPT $  LBSTRESN $ LBSTRESU $ LBTEST $;
   datalines;
   001 Pre 10 mg/dL Normal Hemoglobin
   002 Pre 12 mg/dL Normal Hemoglobin
   003 Pre 15 mg/dL High Hemoglobin
   004 Post 20 mg/dL High Hemoglobin
   005 Pre 80 mg/dL Normal Cholesterol
   006 Pre 90 mg/dL High Cholesterol
   007 Post 70 mg/dL Normal Cholesterol
   008 Post 60 mg/dL Normal Cholesterol
   ;
run;

                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: The data set WORK.SDM_LB has 8 observations and 5 variables.
NOTE: DATA statement used (Total process time):
      real time           0.01 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000024


In [24]:
/* Create Randomized Treatment Data (TRT) */
ods html close;
options nodate nosource notes nonumber;
data sdm_trt;
   input USUBJID $ TRTSDT : yymmdd10. TRTEDT : yymmdd10. TRTGROUP $;
   format TRTSDT TRTEDT yymmdd10.;
   datalines;
   001 2023-01-01 2023-03-01 DrugA
   002 2023-01-05 2023-03-10 DrugB
   003 2023-01-10 2023-03-15 DrugA
   004 2023-01-15 2023-03-20 DrugB
   005 2023-01-20 2023-03-25 DrugA
   006 2023-01-25 2023-03-30 DrugB
   007 2023-01-30 2023-04-01 DrugA
   008 2023-02-05 2023-04-10 DrugB
   ;
run;

                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: The data set WORK.SDM_TRT has 8 observations and 4 variables.
NOTE: DATA statement used (Total process time):
      real time           0.01 seconds
      cpu time            0.00 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000026


In [26]:

/* Create Analysis Dataset for Adverse Events (ADAE) */
ods html close;
options nodate nosource notes nonumber;
data adam_adae;
   set sdm_ae;
   if AESEV = 'Severe' then AESEV = '1';
   else if AESEV = 'Moderate' then AESEV = '2';
   else AESEV = '3';
   /* Create analysis flags for related and severity */
   if AEREL = 'Related' then AERELFL = 'Y';
   else AERELFL = 'N';
run;

                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: There were 8 observations read from the data set WORK.SDM_AE.
NOTE: The data set WORK.ADAM_ADAE has 8 observations and 6 variables.
NOTE: DATA statement used (Total process time):
      real time           0.00 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000028


In [28]:
/* Create Analysis Dataset for Laboratory Results (ADLB) */
ods html close;
options nodate nosource notes nonumber;
data adam_adlb;
   set sdm_lb;
   if LBSTRESN > 15 then LBSTRESU = 'High';
   else LBSTRESU = 'Normal';
   /* Add lab test result flag */
   if LBTEST = 'Hemoglobin' then LBTESTFL = '1';
   else LBTESTFL = '0';
run;

                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: Character values have been converted to numeric values at the places given by: (Line):(Column).
      1393:7   
NOTE: There were 8 observations read from the data set WORK.SDM_LB.
NOTE: The data set WORK.ADAM_ADLB has 8 observations and 6 variables.
NOTE: DATA statement used (Total process time):
      real time           0.01 seconds
      cpu time            0.00 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000030


In [30]:
/* Create Subject-level Analysis Data (ADSL) */
ods html close;
options nodate nosource notes nonumber;
data adam_adsl;
   set sdm_dm;
   /* Gender flag */
   if SEX = 'M' then SEXFL = '1';
   else SEXFL = '0';
   /* Age group flag */
   if AGE < 40 then AGEGRP = 'Young';
   else if AGE >= 40 and AGE < 60 then AGEGRP = 'Middle-aged';
   else AGEGRP = 'Senior';
run;

                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: There were 8 observations read from the data set WORK.SDM_DM.
NOTE: The data set WORK.ADAM_ADSL has 8 observations and 7 variables.
NOTE: DATA statement used (Total process time):
      real time           0.02 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000032


In [32]:
/* Extended Analysis Dataset for Treatment (ADTRT) */
ods html close;
options nodate nosource notes nonumber;
data adam_adtrt;
   set sdm_trt;
   /* Treatment group flag */
   if TRTGROUP = 'DrugA' then TRTFL = '1';
   else if TRTGROUP = 'DrugB' then TRTFL = '2';
run;

                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: There were 8 observations read from the data set WORK.SDM_TRT.
NOTE: The data set WORK.ADAM_ADTRT has 8 observations and 5 variables.
NOTE: DATA statement used (Total process time):
      real time           0.00 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000034


In [34]:
/* Combining Multiple Datasets for a Final ADaM Analysis Dataset */
ods html close;
options nodate nosource notes nonumber;
data adam_final;
   merge adam_adsl(in=a) adam_adae(in=b) adam_adlb(in=c) adam_adtrt(in=d);
   by USUBJID;
   if a and b and c and d;
run;

                                                           The SAS System

NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1

NOTE: There were 8 observations read from the data set WORK.ADAM_ADSL.
NOTE: There were 8 observations read from the data set WORK.ADAM_ADAE.
NOTE: There were 8 observations read from the data set WORK.ADAM_ADLB.
NOTE: There were 8 observations read from the data set WORK.ADAM_ADTRT.
NOTE: The data set WORK.ADAM_FINAL has 8 observations and 21 variables.
NOTE: DATA statement used (Total process time):
      real time           0.01 seconds
      cpu time            0.01 seconds
      

                                                           The SAS System

E3969440A681A2408885998500000036
